# Statistische Analyse

Dieses Notebook wertet `output/statistics.csv` aus und erlaubt:
- Ausschluss bestimmter `Vehicles` und `Scenarios`
- Auswahl beliebiger `Player` ü den Vergleich
- Vergleich der Noten-Häufigkeit ü variable Disziplinen (z. B. `Median`, `P90`, `P95`)


In [28]:
import pandas as pd
import plotly.express as px

pd.set_option('display.max_columns', 50)


## Get the Data

In [29]:
# Pfad zur CSV-Datei
CSV_PATH = 'output/statistics.csv'

df = pd.read_csv(CSV_PATH, encoding='utf-8', encoding_errors='replace')

# Häufige Encoding-Artefakte (z. B. ÃƒÆ’Ã‚Â¼) in String-Spalten bereinigen
def _fix_mojibake(val):
    if not isinstance(val, str):
        return val
    mapping = {
        'ÃƒÆ’Ã‚Â¤': 'ä', 'ÃƒÆ’Ã‚Â¶': 'ÃƒÂ¶', 'ÃƒÆ’Ã‚Â¼': 'ÃƒÂ¼', 'ÃƒÆ’Ã¢â‚¬Å¾': 'Ãƒâ€ž', 'ÃƒÆ’Ã¢â‚¬â€œ': 'Ãƒâ€“', 'ÃƒÆ’Ã…â€œ': 'ÃƒÅ“', 'ÃƒÆ’Ã…Â¸': 'ÃƒÅ¸'
    }
    for bad, good in mapping.items():
        val = val.replace(bad, good)
    return val

#for c in df.select_dtypes(include='object').columns:
#    df[c] = df[c].map(_fix_mojibake)
display(df.head(3))
print(f'Anzahl Datensätze: {len(df)}')
print('Spalten:', ', '.join(df.columns))
print(f'Players: {df["Player"].value_counts()}')


,Dateiname,Mittelwert,Mittelwert_Note,Median,Median_Note,P90,P90_Note,P95,P95_Note,hervorragend,sehr gut,gut,ausreichend,ungenügend,Player,Vehicles,Scenarios
0,20260129_HOME_Extender_BOOSTCLGA_LOADYOUTUBE4K...,20.9,gut,19.0,sehr gut,24.0,gut,29.0,gut,0.0,58.5,39.0,2.0,0.5,Optima_Prio_CLGA,Extender,Youtube 4k
1,20260129_HOME_Extender_BOOSTNOPRIO_LOADOFF.csv,16.9,sehr gut,16.0,sehr gut,20.0,gut,23.0,gut,0.0,88.0,12.0,0.0,0.0,Optima_active,Extender,No Load
2,20260129_HOME_Extender_BOOSTNOPRIO_LOADTCP8PPP...,23.1,gut,22.0,gut,30.0,gut,34.0,gut,0.0,30.2,67.8,2.0,0.0,Optima_active,Extender,TCP


Anzahl Datensätze: 77
Spalten: Dateiname, Mittelwert, Mittelwert_Note, Median, Median_Note, P90, P90_Note, P95, P95_Note, hervorragend, sehr gut, gut, ausreichend, ungenügend, Player, Vehicles, Scenarios
Players: Player
Optima_Prio_CLGA      32
Magenta_Home          24
Optima_deactivated    11
Optima_active         10
Name: count, dtype: int64


## Set the Filters

In [30]:
# Konfiguration
# Diese Listen kannst du direkt anpassen.

# 1) Werte, die NICHT berücksichtigt werden sollen
exclude_vehicles = ["Extender", 'LAN']
exclude_scenarios = ["STEAM"]


# 2) Player, die verglichen werden sollen
selected_players = ['Optima_Prio_CLGA', 'Magenta_Home']

# 3) Disziplinen, deren _Note-Spalten verglichen werden
selected_disciplines = ['Median', 'P90', 'P95']

# 4) Testläufe die verglichen werden sollen
selected_runs = ['20260219','20260210']
#selected_runs = ['20260219']
valid_disciplines = ['Mittelwert', 'Median', 'P90', 'P95']
invalid = [d for d in selected_disciplines if d not in valid_disciplines]
if invalid:
    raise ValueError(f'Ungültige Disziplin(en): {invalid}. Erlaubt: {valid_disciplines}')

for d in selected_disciplines:
    note_col = f'{d}_Note'
    if note_col not in df.columns:
        raise KeyError(f'Spalte fehlt: {note_col}')



## Apply the Filters

In [31]:
# Filter anwenden
df_f = df.copy()

if exclude_vehicles:
    df_f = df_f[~df_f['Vehicles'].isin(exclude_vehicles)]
if exclude_scenarios:
    df_f = df_f[~df_f['Scenarios'].isin(exclude_scenarios)]
if selected_players:
    df_f = df_f[df_f['Player'].isin(selected_players)]
if selected_runs:
    df_0=pd.DataFrame()
    for k in selected_runs:
        df_0=pd.concat([df_0, df_f[df_f['Dateiname'].str.contains(k)]])
        #df_f= df_f[df_f['Dateiname'].str.contains(k)]
    df_f = df_0
    

print(f'Gefilterte Datensätze: {len(df_f)}')
display(df_f[['Player', 'Vehicles', 'Scenarios']].head(16))
print(f'{df_f["Vehicles"].value_counts()},\n{df_f["Scenarios"].value_counts()}')

Gefilterte Datensätze: 16


,Player,Vehicles,Scenarios
69,Optima_Prio_CLGA,WLAN,No Load
70,Optima_Prio_CLGA,WLAN,TCP
71,Optima_Prio_CLGA,WLAN,Youtube 4k
72,Optima_Prio_CLGA,WLAN,Youtube 4k + MagentaTV
73,Magenta_Home,WLAN,No Load
74,Magenta_Home,WLAN,TCP
75,Magenta_Home,WLAN,Youtube 4k
76,Magenta_Home,WLAN,Youtube 4k + MagentaTV
53,Optima_Prio_CLGA,WLAN,No Load
54,Optima_Prio_CLGA,WLAN,TCP


Vehicles
WLAN    16
Name: count, dtype: int64,
Scenarios
No Load                   4
TCP                       4
Youtube 4k                4
Youtube 4k + MagentaTV    4
Name: count, dtype: int64


## Calculate Noten-Häufigkeit pro Player und Disziplin berechnen

In [32]:
# Noten-Häufigkeit pro Player und Disziplin berechnen
grade_order = ['hervorragend', 'sehr gut', 'gut', 'ausreichend', 'ungenügend']

parts = []
for d in selected_disciplines:
    note_col = f'{d}_Note'
    g = (
        df_f.groupby(['Player', note_col], dropna=False)
        .size()
        .reset_index(name='Anzahl')
        .rename(columns={note_col: 'Note'})
    )
    g['Disziplin'] = d
    parts.append(g)

freq = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=['Player', 'Note', 'Anzahl', 'Disziplin'])
freq['Note'] = pd.Categorical(freq['Note'], categories=grade_order, ordered=True)
freq = freq.sort_values(['Disziplin', 'Player', 'Note']).reset_index(drop=True)

display(freq)


,Player,Note,Anzahl,Disziplin
0,Magenta_Home,hervorragend,3,Median
1,Magenta_Home,sehr gut,3,Median
2,Magenta_Home,gut,2,Median
3,Optima_Prio_CLGA,hervorragend,4,Median
4,Optima_Prio_CLGA,sehr gut,4,Median
5,Magenta_Home,hervorragend,1,P90
6,Magenta_Home,sehr gut,3,P90
7,Magenta_Home,gut,4,P90
8,Optima_Prio_CLGA,sehr gut,1,P90
9,Optima_Prio_CLGA,gut,7,P90


## Visualization: isualisierung: Haeufigkeit je Note, getrennt nach Disziplin

In [33]:
# Visualisierung: Haeufigkeit je Note, getrennt nach Disziplin (Plotly)
if freq.empty:
    print('Keine Daten nach Filterung vorhanden.')
else:
    p = freq.copy()
    p['Note'] = p['Note'].astype(str)

    fig = px.bar(
        p,
        x='Note',
        y='Anzahl',
        color='Player',
        facet_col='Disziplin',
        barmode='group',
        category_orders={'Note': grade_order, 'Disziplin': selected_disciplines},
        title='Noten-Haeufigkeit je Disziplin'
    )

    fig.update_xaxes(categoryorder='array', categoryarray=grade_order, tickangle=30)
    fig.update_yaxes(title_text='Haeufigkeit')
    fig.for_each_annotation(lambda a: a.update(text=a.text.replace('Disziplin=', 'Disziplin: ')))
    fig.update_layout(legend_title_text='Player')
    fig.show()


## Bester Player je Diziplin (Median, P90, P95, ...)

In [34]:
# Optional: "Wer hat besser abgeschlossen?" ÃƒÂ¼ber Noten-Score je Disziplin
# Niedriger Score ist besser.
grade_score = {'hervorragend': 1, 'sehr gut': 2, 'gut': 3, 'ausreichend': 4, 'ungenÃƒÂ¼gend': 5}

ranking_rows = []
for d in selected_disciplines:
    note_col = f'{d}_Note'
    tmp = df_f[['Player', note_col]].copy().rename(columns={note_col: 'Note'})
    tmp['Score'] = tmp['Note'].map(grade_score)
    r = tmp.groupby('Player', as_index=False)['Score'].mean().sort_values('Score')
    r['Disziplin'] = d
    ranking_rows.append(r)

ranking = pd.concat(ranking_rows, ignore_index=True) if ranking_rows else pd.DataFrame(columns=['Player', 'Score', 'Disziplin'])
ranking = ranking[['Disziplin', 'Player', 'Score']].sort_values(['Disziplin', 'Score'])

display(ranking)

if not ranking.empty:
    winners = ranking.groupby('Disziplin', as_index=False).first()
    print('Beste Player je Disziplin (nach ÃƒËœ-Noten-Score):')
    display(winners)


,Disziplin,Player,Score
0,Median,Optima_Prio_CLGA,1.500
1,Median,Magenta_Home,1.875
2,P90,Magenta_Home,2.375
3,P90,Optima_Prio_CLGA,2.875
4,P95,Magenta_Home,2.750
5,P95,Optima_Prio_CLGA,3.125


Beste Player je Disziplin (nach ÃƒËœ-Noten-Score):


,Disziplin,Player,Score
0,Median,Optima_Prio_CLGA,1.500
1,P90,Magenta_Home,2.375
2,P95,Magenta_Home,2.750


## Direkter Szenario-Vergleich (kleinerer Zahlenwert gewinnt)

In [ ]:
# Direkter Szenario-Vergleich (kleinerer Zahlenwert gewinnt)
if len(selected_players) != 2:
    raise ValueError('Fuer den direkten Vergleich bitte genau 2 Player in selected_players setzen.')

p1, p2 = selected_players
vergleich_rows = []

for d in selected_disciplines:
    metric_col = d
    if metric_col not in df_f.columns:
        raise KeyError(f'Spalte fehlt fuer Disziplin {d}: {metric_col}')

    m = (
        df_f[df_f['Player'].isin([p1, p2])]
        .groupby(['Scenarios', 'Player'], as_index=False)[metric_col]
        .median()
    )

    pivot = m.pivot(index='Scenarios', columns='Player', values=metric_col).reset_index()
    if p1 not in pivot.columns or p2 not in pivot.columns:
        continue

    pivot = pivot.dropna(subset=[p1, p2]).copy()
    if pivot.empty:
        continue

    pivot['Disziplin'] = d
    pivot['Gewinner'] = pivot.apply(
        lambda r: p1 if r[p1] < r[p2] else (p2 if r[p2] < r[p1] else 'Unentschieden'),
        axis=1
    )
    pivot['Differenz_ms'] = (pivot[p1] - pivot[p2]).abs()

    vergleich_rows.append(
        pivot[['Disziplin', 'Scenarios', p1, p2, 'Gewinner', 'Differenz_ms']]
        .rename(columns={p1: f'{p1}_ms', p2: f'{p2}_ms'})
    )

vergleich = pd.concat(vergleich_rows, ignore_index=True) if vergleich_rows else pd.DataFrame()

if vergleich.empty:
    print('Keine vergleichbaren Szenario-Daten fuer die beiden Player gefunden.')
else:
    display(vergleich.sort_values(['Disziplin', 'Scenarios']).reset_index(drop=True))

    print('Beispiel-Ausgabe je Szenario:')
    for _, r in vergleich.sort_values(['Disziplin', 'Scenarios']).iterrows():
        print(f"{r['Disziplin']} {p1}: {r[f'{p1}_ms']:.2f} ms")
        print(f"{r['Disziplin']} {p2}: {r[f'{p2}_ms']:.2f} ms")
        print(f"{r['Gewinner']} hat gewonnen.")
        print('-' * 40)

    win_freq = (
        vergleich[vergleich['Gewinner'] != 'Unentschieden']
        .groupby(['Disziplin', 'Gewinner'])
        .size()
        .reset_index(name='Siege')
        .sort_values(['Disziplin', 'Siege'], ascending=[True, False])
    )

    print('Haeufigkeit der Siege je Player und Disziplin:')
    display(win_freq)

    win_total = (
        vergleich[vergleich['Gewinner'] != 'Unentschieden']['Gewinner']
        .value_counts()
        .rename_axis('Player')
        .reset_index(name='Siege_gesamt')
    )

    print('Gesamte Siege ueber alle Disziplinen/Szenarien:')
    display(win_total)


Player,Disziplin,Scenarios,Optima_Prio_CLGA_ms,Magenta_Home_ms,Gewinner,Differenz_ms
0,Median,No Load,10.00,9.0,Magenta_Home,1.00
1,Median,TCP,12.25,30.0,Optima_Prio_CLGA,17.75
2,Median,Youtube 4k,11.00,10.0,Magenta_Home,1.00
3,Median,Youtube 4k + MagentaTV,11.00,12.5,Optima_Prio_CLGA,1.50
4,P90,No Load,18.50,10.0,Magenta_Home,8.50
5,P90,TCP,23.00,34.0,Optima_Prio_CLGA,11.00
6,P90,Youtube 4k,25.50,21.0,Magenta_Home,4.50
7,P90,Youtube 4k + MagentaTV,24.00,22.0,Magenta_Home,2.00
8,P95,No Load,24.00,10.0,Magenta_Home,14.00
9,P95,TCP,31.00,35.0,Optima_Prio_CLGA,4.00


Beispiel-Ausgabe je Szenario:
Median Optima_Prio_CLGA: 10.00 ms
Median Magenta_Home: 9.00 ms
Magenta_Home hat gewonnen.
----------------------------------------
Median Optima_Prio_CLGA: 12.25 ms
Median Magenta_Home: 30.00 ms
Optima_Prio_CLGA hat gewonnen.
----------------------------------------
Median Optima_Prio_CLGA: 11.00 ms
Median Magenta_Home: 10.00 ms
Magenta_Home hat gewonnen.
----------------------------------------
Median Optima_Prio_CLGA: 11.00 ms
Median Magenta_Home: 12.50 ms
Optima_Prio_CLGA hat gewonnen.
----------------------------------------
P90 Optima_Prio_CLGA: 18.50 ms
P90 Magenta_Home: 10.00 ms
Magenta_Home hat gewonnen.
----------------------------------------
P90 Optima_Prio_CLGA: 23.00 ms
P90 Magenta_Home: 34.00 ms
Optima_Prio_CLGA hat gewonnen.
----------------------------------------
P90 Optima_Prio_CLGA: 25.50 ms
P90 Magenta_Home: 21.00 ms
Magenta_Home hat gewonnen.
----------------------------------------
P90 Optima_Prio_CLGA: 24.00 ms
P90 Magenta_Home: 22.0

,Disziplin,Gewinner,Siege
0,Median,Magenta_Home,2
1,Median,Optima_Prio_CLGA,2
2,P90,Magenta_Home,3
3,P90,Optima_Prio_CLGA,1
4,P95,Magenta_Home,3
5,P95,Optima_Prio_CLGA,1


Gesamte Siege ueber alle Disziplinen/Szenarien:


,Player,Siege_gesamt
0,Magenta_Home,8
1,Optima_Prio_CLGA,4


: 